# NBA Advanced Stats Scraper

Scrapes BoxScoreAdvancedV3 data for all games in the database.

**Features:**
- Resumable: Tracks progress in `scrape_advanced_stats_progress` table
- Crash-safe: Commits every 50 games
- Rate-limited: Random delays + long pauses to avoid API bans
- DNP handling: Flags players who didn't play

**Before running:**
1. Execute `advanced_stats_schema.sql` in Supabase to create tables
2. Ensure `.env` file has `DATABASE_URL`

In [58]:
# =============================================================================
# Imports and Setup
# =============================================================================

from requests.exceptions import ReadTimeout, ConnectionError
from sqlalchemy import create_engine, text, inspect
from dotenv import load_dotenv
from datetime import datetime
import pandas as pd
import numpy as np
import random
import time
import json
import re
import os

from nba_api.stats.endpoints import boxscoreadvancedv3

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

print("Imports complete")

Imports complete


In [59]:
# =============================================================================
# Configuration
# =============================================================================

# Scraper settings
BATCH_SIZE = 50                    # Commit to DB every N games
SHORT_DELAY_MIN = 0.1              # Min seconds between API calls
SHORT_DELAY_MAX = 1.5              # Max seconds between API calls  
LONG_PAUSE_EVERY = 100             # Long pause every N games
LONG_PAUSE_MIN = 30                # Min seconds for long pause
LONG_PAUSE_MAX = 100               # Max seconds for long pause
ERROR_RETRY_DELAY = 120            # Seconds to wait after error
MAX_RETRIES = 3                    # Max retries per game before skipping

# Database connection
load_dotenv()
DATABASE_URL = os.getenv("DATABASE_URL")

if not DATABASE_URL:
    raise ValueError("DATABASE_URL not found in environment. Check your .env file.")

engine = create_engine(DATABASE_URL)
print(f"Connected to database")

# Test connection
with engine.connect() as conn:
    result = conn.execute(text("SELECT 1"))
    print("Database connection verified ✓")

Connected to database
Database connection verified ✓


In [60]:
# =============================================================================
# Helper Functions
# =============================================================================

def camel_to_snake(name: str) -> str:
    """
    Convert camelCase to snake_case.
    e.g., 'offensiveRating' -> 'offensive_rating'
          'PIE' -> 'pie'
    """
    # Handle acronyms like PIE
    s1 = re.sub('(.)([A-Z][a-z]+)', r'\1_\2', name)
    return re.sub('([a-z0-9])([A-Z])', r'\1_\2', s1).lower()


def parse_minutes(minutes_str) -> float:
    """
    Parse minutes from various formats to float.
    
    Handles:
    - 'MM:SS' format (e.g., '28:17' -> 28.283)
    - 'PT28M17.00S' ISO format
    - Already numeric values
    - None/empty/NaN -> 0.0
    """
    if pd.isna(minutes_str) or minutes_str == '' or minutes_str is None:
        return 0.0
    
    # Already numeric
    if isinstance(minutes_str, (int, float)):
        return float(minutes_str)
    
    minutes_str = str(minutes_str)
    
    # MM:SS format (BoxScoreAdvancedV3)
    if ':' in minutes_str and 'PT' not in minutes_str:
        try:
            parts = minutes_str.split(':')
            mins = int(parts[0])
            secs = int(parts[1]) if len(parts) > 1 else 0
            return round(mins + secs / 60, 3)
        except (ValueError, IndexError):
            return 0.0
    
    # ISO 8601 format (PT28M17.00S)
    if 'PT' in minutes_str:
        try:
            match = re.match(r'PT(\d+)M([\d.]+)S', minutes_str)
            if match:
                mins = int(match.group(1))
                secs = float(match.group(2))
                return round(mins + secs / 60, 3)
        except (ValueError, AttributeError):
            return 0.0
    
    return 0.0


def determine_dnp(row: pd.Series) -> bool:
    """
    Determine if a player did not play.
    
    DNP if:
    - 'comment' field contains text (usually DNP reason)
    - minutes is 0 or null
    """
    comment = row.get('comment', '')
    minutes = row.get('minutes', 0)
    
    # Has a comment (DNP reason)
    if pd.notna(comment) and str(comment).strip() != '':
        return True
    
    # Zero or null minutes
    if pd.isna(minutes) or minutes == 0:
        return True
    
    return False


def transform_player_df(df: pd.DataFrame) -> pd.DataFrame:
    """
    Transform player DataFrame from API format to database format.
    """
    if df.empty:
        return df
    
    # Rename columns: camelCase -> snake_case
    df = df.rename(columns={col: camel_to_snake(col) for col in df.columns})
    
    # Parse minutes
    if 'minutes' in df.columns:
        df['minutes'] = df['minutes'].apply(parse_minutes)
    
    # Add DNP flag
    df['did_not_play'] = df.apply(determine_dnp, axis=1)
    
    # Add timestamp
    df['created_at'] = datetime.now()
    
    return df


def transform_team_df(df: pd.DataFrame) -> pd.DataFrame:
    """
    Transform team DataFrame from API format to database format.
    """
    if df.empty:
        return df
    
    # Rename columns: camelCase -> snake_case
    df = df.rename(columns={col: camel_to_snake(col) for col in df.columns})
    
    # Parse minutes
    if 'minutes' in df.columns:
        df['minutes'] = df['minutes'].apply(parse_minutes)
    
    # Add timestamp
    df['created_at'] = datetime.now()
    
    return df


def rate_limit(game_num: int):
    """
    Apply rate limiting between API calls.
    Long pause every LONG_PAUSE_EVERY games.
    """
    if game_num > 0 and game_num % LONG_PAUSE_EVERY == 0:
        pause = round(random.uniform(LONG_PAUSE_MIN, LONG_PAUSE_MAX), 1)
        print(f"\n  ⏸️  Long pause: {pause}s (every {LONG_PAUSE_EVERY} games)\n")
        time.sleep(pause)
    else:
        delay = round(random.uniform(SHORT_DELAY_MIN, SHORT_DELAY_MAX), 2)
        time.sleep(delay)


print("Helper functions loaded ✓")

# Test camel_to_snake
test_cases = ['offensiveRating', 'PIE', 'assistToTurnover', 'gameId', 'estimatedOffensiveRating']
print("\nColumn name transformation test:")
for tc in test_cases:
    print(f"  {tc} -> {camel_to_snake(tc)}")

Helper functions loaded ✓

Column name transformation test:
  offensiveRating -> offensive_rating
  PIE -> pie
  assistToTurnover -> assist_to_turnover
  gameId -> game_id
  estimatedOffensiveRating -> estimated_offensive_rating


In [61]:
# =============================================================================
# Get Game IDs to Scrape
# =============================================================================

def get_all_game_ids(engine) -> list:
    """
    Get all distinct game IDs from team_game_stats.
    """
    query = """
        SELECT DISTINCT game_id 
        FROM team_game_stats 
        ORDER BY game_id
    """
    df = pd.read_sql(query, engine)
    return df['game_id'].tolist()


def get_scraped_game_ids(engine) -> set:
    """Get game IDs that have already been scraped."""
    inspector = inspect(engine)
    
    if 'advanced_player_game_stats' in inspector.get_table_names():
        query = "SELECT DISTINCT game_id FROM advanced_player_game_stats"
        df = pd.read_sql(query, engine)
        return set(df['game_id'].tolist())
    
    return set()


# Get games
print("Fetching game IDs...")
all_game_ids = get_all_game_ids(engine)
print(f"  Total games in database: {len(all_game_ids)}")

print("\nChecking progress...")
scraped_game_ids = get_scraped_game_ids(engine)
print(f"  Already scraped: {len(scraped_game_ids)}")

# Filter to remaining games
games_to_scrape = [g for g in all_game_ids if g not in scraped_game_ids]
print(f"  Remaining to scrape: {len(games_to_scrape)}")

# Estimate time
avg_time_per_game = (SHORT_DELAY_MIN + SHORT_DELAY_MAX) / 2 + 0.5  # API call time
long_pauses = len(games_to_scrape) // LONG_PAUSE_EVERY
estimated_hours = (len(games_to_scrape) * avg_time_per_game + long_pauses * (LONG_PAUSE_MIN + LONG_PAUSE_MAX) / 2) / 3600
print(f"\n⏱️  Estimated time: {estimated_hours:.1f} hours")

Fetching game IDs...
  Total games in database: 19118

Checking progress...
  Already scraped: 10950
  Remaining to scrape: 8168

⏱️  Estimated time: 4.4 hours


In [62]:
# =============================================================================
# Single Game Scraper (for testing)
# =============================================================================

def scrape_single_game(game_id: str) -> tuple:
    """
    Scrape advanced stats for a single game.
    
    Returns:
        (player_df, team_df) - Both transformed and ready for DB insertion
    """
    box_adv = boxscoreadvancedv3.BoxScoreAdvancedV3(game_id=game_id)
    dfs = box_adv.get_data_frames()
    
    # dfs[0] = players, dfs[1] = teams
    player_df = transform_player_df(dfs[0].copy())
    team_df = transform_team_df(dfs[1].copy())
    
    return player_df, team_df


# Test with first game
if games_to_scrape:
    test_game_id = games_to_scrape[0]
    print(f"Testing with game: {test_game_id}")
    
    try:
        player_df, team_df = scrape_single_game(test_game_id)
        print(f"\nPlayer DataFrame:")
        print(f"  Shape: {player_df.shape}")
        print(f"  Columns: {list(player_df.columns)}")
        print(f"  DNP players: {player_df['did_not_play'].sum()}")
        
        print(f"\nTeam DataFrame:")
        print(f"  Shape: {team_df.shape}")
        print(f"  Columns: {list(team_df.columns)}")
        
        print("\n✓ Test successful")
    except Exception as e:
        print(f"\n❌ Test failed: {e}")
else:
    print("No games to scrape - all done!")

Testing with game: 0021700751

❌ Test failed: HTTPSConnectionPool(host='stats.nba.com', port=443): Read timed out. (read timeout=30)


In [57]:
# =============================================================================
# SINGLE MACHINE SCRAPER (Consolidated & Crash-Proof)
# =============================================================================

from sqlalchemy import text

# --- 1. Robust Save Function (Ignores Duplicates) ---
def save_batch(player_batch: pd.DataFrame, team_batch: pd.DataFrame, 
               progress_batch: list, engine):
    """
    Saves data. If a batch contains duplicates, it catches the error 
    and prints a warning instead of crashing the script.
    """
    # Save Stats
    try:
        if not player_batch.empty:
            player_batch.to_sql('advanced_player_game_stats', engine, if_exists='append', index=False)
        if not team_batch.empty:
            team_batch.to_sql('advanced_team_game_stats', engine, if_exists='append', index=False)
    except Exception as e:
        if "UniqueViolation" in str(e) or "duplicate key" in str(e):
             print(f"  ⚠️  Stats duplicate detected in batch. Ignoring.")
        else:
             print(f"  ⚠️  Stats save error: {e}")

    # Save Progress Log
    try:
        if progress_batch:
            progress_df = pd.DataFrame(progress_batch)
            progress_df.to_sql('scrape_advanced_stats_progress', engine, if_exists='append', index=False)
    except Exception as e:
        if "UniqueViolation" in str(e) or "duplicate key" in str(e):
            print(f"  ⚠️  Progress log duplicate detected. Ignoring.")
        else:
            print(f"  ⚠️  Could not save progress log: {e}")

# --- 2. Pre-check Helper ---
def check_game_exists(game_id, engine):
    """Checks if a game exists in the DB immediately before scraping."""
    try:
        query = text("SELECT 1 FROM advanced_player_game_stats WHERE game_id = :gid LIMIT 1")
        with engine.connect() as conn:
            result = conn.execute(query, {"gid": game_id}).fetchone()
            return result is not None
    except Exception:
        return False

# --- 3. Get Work List (ALL Games) ---
print("Fetching game list...")
all_game_ids = get_all_game_ids(engine)
scraped_game_ids = get_scraped_game_ids(engine)

# Simple subtraction: All games minus what we have = what we need
games_to_scrape = [g for g in all_game_ids if g not in scraped_game_ids]
total_games = len(games_to_scrape)

print(f"Total games in DB: {len(all_game_ids)}")
print(f"Already scraped:   {len(scraped_game_ids)}")
print(f"Remaining to do:   {total_games}")
print(f"{'='*60}\n")

# --- 4. Main Loop ---
player_batch = pd.DataFrame()
team_batch = pd.DataFrame()
progress_batch = []
games_processed = 0
games_skipped = 0

for i, game_id in enumerate(games_to_scrape, 1):
    
    # Double check: did we just scrape this or did it appear in DB?
    if check_game_exists(game_id, engine):
        print(f"[{i}/{total_games}] Game {game_id} already exists. Skipping.")
        games_skipped += 1
        continue

    retries = 0
    success = False
    
    while retries < MAX_RETRIES and not success:
        try:
            print(f"[{i}/{total_games}] Scraping {game_id}...", end="")
            
            # Scrape
            player_df, team_df = scrape_single_game(game_id)
            
            # Add to batch
            player_batch = pd.concat([player_batch, player_df], ignore_index=True)
            team_batch = pd.concat([team_batch, team_df], ignore_index=True)
            progress_batch.append({
                'game_id': game_id,
                'scraped_at': datetime.now(),
                'player_rows_inserted': len(player_df),
                'team_rows_inserted': len(team_df),
                'status': 'success'
            })
            
            print(f" ✓ ({len(player_df)} players)")
            success = True
            games_processed += 1
            
            # Save Batch
            if games_processed % BATCH_SIZE == 0:
                print(f"  💾 Saving batch of {BATCH_SIZE}...")
                save_batch(player_batch, team_batch, progress_batch, engine)
                # Reset accumulators
                player_batch = pd.DataFrame()
                team_batch = pd.DataFrame()
                progress_batch = []
            
            rate_limit(games_processed)

        except Exception as e:
            retries += 1
            print(f"\n  ⚠️  Error ({retries}/{MAX_RETRIES}): {e}")
            if retries < MAX_RETRIES:
                time.sleep(ERROR_RETRY_DELAY)
            else:
                print("  ❌ Failed. Skipping.")
                # Optional: Log failure
                progress_batch.append({
                    'game_id': game_id,
                    'scraped_at': datetime.now(),
                    'player_rows_inserted': 0, 
                    'team_rows_inserted': 0,
                    'status': 'failed'
                })

# Final Save
if not player_batch.empty or progress_batch:
    print("💾 Saving final batch...")
    save_batch(player_batch, team_batch, progress_batch, engine)

print(f"\n✅ DONE. Processed: {games_processed}, Skipped: {games_skipped}")

Fetching game list...
Total games in DB: 19118
Already scraped:   10950
Remaining to do:   8168

[1/8168] Scraping 0021700751...

KeyboardInterrupt: 

In [ ]:
# =============================================================================
# Verification Queries
# =============================================================================

print("Checking scrape results...\n")

# Row counts
player_count = pd.read_sql("SELECT COUNT(*) as cnt FROM advanced_player_game_stats", engine)['cnt'].iloc[0]
team_count = pd.read_sql("SELECT COUNT(*) as cnt FROM advanced_team_game_stats", engine)['cnt'].iloc[0]
progress_count = pd.read_sql("SELECT COUNT(*) as cnt FROM scrape_advanced_stats_progress WHERE status = 'success'", engine)['cnt'].iloc[0]

print(f"Player stats rows: {player_count:,}")
print(f"Team stats rows: {team_count:,}")
print(f"Games successfully scraped: {progress_count:,}")

# Failed games
failed_games = pd.read_sql("""
    SELECT game_id, scraped_at 
    FROM scrape_advanced_stats_progress 
    WHERE status = 'failed'
    ORDER BY scraped_at DESC
""", engine)

if len(failed_games) > 0:
    print(f"\n⚠️  Failed games: {len(failed_games)}")
    print(failed_games.head(10))
else:
    print(f"\n✅ No failed games")

# DNP distribution
dnp_dist = pd.read_sql("""
    SELECT did_not_play, COUNT(*) as count
    FROM advanced_player_game_stats
    GROUP BY did_not_play
""", engine)
print(f"\nDNP distribution:")
print(dnp_dist)

In [ ]:
# =============================================================================
# Retry Failed Games (Optional)
# =============================================================================

# Get failed games
failed_games = pd.read_sql("""
    SELECT game_id 
    FROM scrape_advanced_stats_progress 
    WHERE status = 'failed'
""", engine)

if len(failed_games) > 0:
    failed_game_ids = failed_games['game_id'].tolist()
    print(f"Found {len(failed_game_ids)} failed games to retry")
    
    # First, delete the failed progress records so they can be re-inserted
    # Uncomment to retry:
    # with engine.connect() as conn:
    #     conn.execute(text("DELETE FROM scrape_advanced_stats_progress WHERE status = 'failed'"))
    #     conn.commit()
    # 
    # processed, failed = run_scraper(failed_game_ids, engine)
else:
    print("No failed games to retry")

In [ ]:
# =============================================================================
# Sample Data Preview
# =============================================================================

print("Sample player advanced stats:")
sample_player = pd.read_sql("""
    SELECT game_id, person_id, first_name, family_name, team_tricode,
           minutes, offensive_rating, defensive_rating, net_rating,
           usage_percentage, true_shooting_percentage, pie, did_not_play
    FROM advanced_player_game_stats
    WHERE did_not_play = FALSE
    LIMIT 10
""", engine)
display(sample_player)

print("\nSample team advanced stats:")
sample_team = pd.read_sql("""
    SELECT game_id, team_id, team_tricode,
           minutes, offensive_rating, defensive_rating, net_rating,
           pace, possessions, pie
    FROM advanced_team_game_stats
    LIMIT 10
""", engine)
display(sample_team)